### Libraries


In [1]:
import pandas as pd
import sqlite3
import json


### Create Customer Data

In [2]:
customers = pd.DataFrame({
    "CustomerID": [
"C001", "C002", "C003", "C004",
        "C004", "C005", "C006", "C007",
        "C008", "C009", "C010"
    ],
    "CustomerName": [
        "Amina", "Brian", "Carol", "David",
        "David", "Esther", "Faith", "George",
        "Halima", "Ian", "Jane"
    ],
    "Segment": [
        "Retail", "Corporate", "Retail", "SME",
        "SME", "Corporate", "SME", "Retail",
        "Corporate", "Retail", "SME"
    ],
    "RegionCode": [
        "NRB", "MSA", "NRB", "KSM",
        "KSM", "NKR", "NRB", "MSA",
        "KSM", "NKR", "NRB"
] })
customers

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM
5,C005,Esther,Corporate,NKR
6,C006,Faith,SME,NRB
7,C007,George,Retail,MSA
8,C008,Halima,Corporate,KSM
9,C009,Ian,Retail,NKR


In [3]:
# Save the DataFrame to a CSV file
customers.to_csv("customers.csv", index=False)

### Create the Orders Data

In [4]:
orders = pd.DataFrame({
    "OrderID": [
        "O001", "O002", "O003", "O004", "O005",
        "O006", "O007", "O008", "O009", "O010",
        "O011", "O012", "O013", "O014", "O015"
    ],
    "CustomerID": [
        "C001", "C002", "C001", "C003", "C004",
        "C005", "C006", "C002", "C007", "C008",
        "C009", "C010", "C006", "C003", "C999"
    ],
    "OrderAmount": [
        4500, 12500, 3200, 6800, 9100,
        15000, 7600, 8300, 2500, 11000,
        5400, 6200, 4300, 7200, 9900
    ],
    "ProductCategory": [
        "Electronics", "Office", "Home", "Electronics", "Office",
        "Electronics", "Home", "Office", "Home", "Electronics",
        "Office", "Home", "Electronics", "Office", "Electronics"
] })
orders


,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office
5,O006,C005,15000,Electronics
6,O007,C006,7600,Home
7,O008,C002,8300,Office
8,O009,C007,2500,Home
9,O010,C008,11000,Electronics


In [5]:
# Save the DataFrame to a CSV file
orders.to_csv("orders.csv", index=False)

### Create the JSON Region Lookup

In [6]:
regions = [
    {"RegionCode": "NRB", "RegionName": "Nairobi"},
    {"RegionCode": "MSA", "RegionName": "Mombasa"},
    {"RegionCode": "KSM", "RegionName": "Kisumu"},
    {"RegionCode": "NKR", "RegionName": "Nakuru"}
]
with open("regions.json", "w") as file:
    json.dump(regions, file, indent=4)

### Inspect the sources

In [7]:
# Load the CSV files into DataFrames
customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')

In [8]:
customers.head()    

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


##### Q1 - Each row represents a customer, with their unique ID, name, segment, and region code.

In [9]:
orders.head()

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office


##### Q2 - Each row represents an order from a customer, with the order ID, customer ID, order amount, and product category.

In [10]:
customers.shape

(11, 4)

In [11]:
orders.shape

(15, 4)

##### Q3 - The two datasets have different aggregation grains. The customers dataset is one row per customer while the orders dataset is one row per order and a customer can appear many times.

### Detect key problems

In [12]:
# check for duplicates in customers DataFrame
customers['CustomerID'].duplicated().sum()


np.int64(1)

In [13]:
customers[
    customers["CustomerID"].duplicated(keep=False)
]

,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


##### Q4 - A join matches every order row to every customer row with the same CustomerID. If C004 appears twice in customers, each C004 order matches twice, so the result will have two copies of it.

In [14]:
# Check unmatched CustomerIDs
set(orders["CustomerID"]) - set(customers["CustomerID"])

{'C999'}

##### Q5 - C999 appears in orders but not in customers

##### Q6 - An unmatched record is a symptom of a problem and deleting it hides the cause while altering results

### Create the SQLite Database

In [15]:
# Connect to the SQLite database
conn = sqlite3.connect('retail_lab.db')

In [16]:
# connect the tables to the database
customers.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

orders.to_sql(
    "orders",
    conn,
    if_exists="replace",
index=False
)

15

In [17]:
# test the database
query = """
SELECT * 
FROM customers
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


### SELECT and WHERE

In [18]:
# basic select
query = """
SELECT OrderID, CustomerID, OrderAmount
FROM orders;
"""

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount
0,O001,C001,4500
1,O002,C002,12500
2,O003,C001,3200
3,O004,C003,6800
4,O005,C004,9100
5,O006,C005,15000
6,O007,C006,7600
7,O008,C002,8300
8,O009,C007,2500
9,O010,C008,11000


In [19]:
# a query that returns only customers in the Retail segment
query = """
SELECT *
FROM customers
WHERE Segment = 'Retail';
"""

pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


In [20]:
# a query that returns orders with OrderAmount greater than KSh 8,000
query = """
SELECT *
FROM orders
WHERE OrderAmount > 8000;
""" 

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O002,C002,12500,Office
1,O005,C004,9100,Office
2,O006,C005,15000,Electronics
3,O008,C002,8300,Office
4,O010,C008,11000,Electronics
5,O015,C999,9900,Electronics


In [21]:
# query that returns only Electronics orders
query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics';
""" 

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O004,C003,6800,Electronics
2,O006,C005,15000,Electronics
3,O010,C008,11000,Electronics
4,O013,C006,4300,Electronics
5,O015,C999,9900,Electronics


In [22]:
# query that returns Electronics orders worth more than KSh 8,000.
query = """
SELECT *
FROM orders
WHERE ProductCategory = 'Electronics' AND OrderAmount > 8000;
"""

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O006,C005,15000,Electronics
1,O010,C008,11000,Electronics
2,O015,C999,9900,Electronics


### GROUP BY and Aggregate Functions


In [23]:
# Total sales
query = """
SELECT SUM(OrderAmount) AS TotalSales
FROM orders;
"""

pd.read_sql_query(query, conn)

,TotalSales
0,113500


In [24]:
# Sales by Product Category
query = """
SELECT ProductCategory, SUM(OrderAmount) AS TotalSales
FROM orders
GROUP BY ProductCategory;
"""

pd.read_sql_query(query, conn)

,ProductCategory,TotalSales
0,Electronics,51500
1,Home,19500
2,Office,42500


In [25]:
# a query that returns the number of orders, total sales, and average order amount value for each product category
query = """
SELECT ProductCategory, COUNT(OrderID) AS NumberOfOrders, SUM(OrderAmount) AS TotalSales, AVG(OrderAmount) AS AverageOrderAmountValue
FROM orders
GROUP BY ProductCategory;
"""

pd.read_sql_query(query, conn)

,ProductCategory,NumberOfOrders,TotalSales,AverageOrderAmountValue
0,Electronics,6,51500,8583.333333
1,Home,4,19500,4875.000000
2,Office,5,42500,8500.000000


##### Q7 
##### - The electonics product category performed the best having the most orders and accumulating the highest Sales Total.
##### - The Home product category performed the worst having the least orders and lowest Sales Total.

### JOIN Customers and Orders

In [26]:
#connect order transactions to customer information.
query = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    o.OrderAmount
FROM orders o
LEFT JOIN customers c
    ON o.CustomerID = c.CustomerID;
"""
joined = pd.read_sql_query(query, conn)
joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


In [27]:
# number of rows
len(orders)


15

In [28]:
# number of rows
len(joined)

16

##### Q8 - The row counts aren't equal, joined has 1 more row than orders. There is an extra row because of the duplicate customer row in the customers dataset hence the resulting join has 2 copies

##### Q9 - None and None 

### Reconcile the Totals

In [29]:
# Total before join
orders['OrderAmount'].sum()

np.int64(113500)

In [30]:
# Total after join
joined['OrderAmount'].sum()

np.int64(122600)

##### Q10 - The 2 sales totals arent equal, the total be for the join is 113500 and 122600 after

##### Q11 - Customer C004 caused the duplication

##### Q12 - Because a join can change your numbers without raising any error like when you have duplicate rows or dropped rows, and reconciliation is how you prove it didn't

### Correct the Duplicate-Key Problem

In [31]:
# dropping duplicates
customers_clean = customers.drop_duplicates(subset='CustomerID', keep='first')
customers_clean

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
5,C005,Esther,Corporate,NKR
6,C006,Faith,SME,NRB
7,C007,George,Retail,MSA
8,C008,Halima,Corporate,KSM
9,C009,Ian,Retail,NKR
10,C010,Jane,SME,NRB


In [32]:
# save the cleaned customers DF to database
customers_clean.to_sql(
    "customers_clean",
    conn,
    if_exists="replace",
    index=False
)

10

In [33]:
# join the cleaned customers table with the orders table
query = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    o.OrderAmount
FROM orders o
LEFT JOIN customers_clean c
    ON o.CustomerID = c.CustomerID;
"""
clean_joined = pd.read_sql_query(query, conn)
clean_joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O006,C005,Esther,Corporate,15000
6,O007,C006,Faith,SME,7600
7,O008,C002,Brian,Corporate,8300
8,O009,C007,George,Retail,2500
9,O010,C008,Halima,Corporate,11000


In [34]:
# adding cleaned join  to the database
clean_joined.to_sql(
    "clean_joined",
    conn,
    if_exists="replace",
    index=False
)

15

In [35]:
# checking if row counnts are equal
len(orders), len(clean_joined)

(15, 15)

In [36]:
# checking if total sales are equal
orders['OrderAmount'].sum(), clean_joined['OrderAmount'].sum()  

(np.int64(113500), np.int64(113500))

##### Q13 - The corrected join has preserved both row and total sales values at 15 and 113,500 respectively. This is because we removed the duplicated rows which insured that the join didnt have 2 copies of the same record.

### Customer Segment KPIs

In [37]:
query = """
SELECT Segment, COUNT(OrderID) AS OrderCount, SUM(OrderAmount) AS TotalOrderAmount, AVG(OrderAmount) AS AverageOrderAmount
FROM clean_joined   
GROUP BY Segment;
"""

pd.read_sql_query(query, conn)  

,Segment,OrderCount,TotalOrderAmount,AverageOrderAmount
0,None,1,9900,9900.000000
1,Corporate,4,46800,11700.000000
2,Retail,6,29600,4933.333333
3,SME,4,27200,6800.000000


##### Q14
##### - Corporate customers are the highest-value segment, they averag 11,700 per order and contributing the largest total 46,800 from only 4 orders.
##### - Retail places the most orders 6 but at a low average of about 4,933, so its 29,600 total comes from volume rather than order size. 
##### - SME sits in the middle with 4 orders averaging 6,800 with a total of 27,200 total.

### Merge the JSON Region Lookup

In [38]:
# loading the regions JSON file into a DataFrame
regions_df = pd.read_json('regions.json')
regions_df

,RegionCode,RegionName
0,NRB,Nairobi
1,MSA,Mombasa
2,KSM,Kisumu
3,NKR,Nakuru


In [39]:
# merging the regions with customers
customers_region = customers_clean.merge(regions_df, on='RegionCode', how='left')
customers_region.head()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru


In [40]:
customers_region

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru
5,C006,Faith,SME,NRB,Nairobi
6,C007,George,Retail,MSA,Mombasa
7,C008,Halima,Corporate,KSM,Kisumu
8,C009,Ian,Retail,NKR,Nakuru
9,C010,Jane,SME,NRB,Nairobi


#### Q15 - Every customer has a region name

##### Q16 - It would produce a NaN

### Regional Sales Analysis

In [41]:
# adding the customers with regions names to the database
customers_region.to_sql(
    "customers_region",
    conn,
    if_exists="replace",
    index=False
)

10

In [42]:
#joining the customers with regions names to the orders table
query = """
SELECT
    o.OrderID,
    o.CustomerID,
    c.CustomerName,
    c.Segment,
    c.RegionName,
    o.OrderAmount
FROM orders o
LEFT JOIN customers_region c
    ON o.CustomerID = c.CustomerID;
"""
final_joined = pd.read_sql_query(query, conn)
final_joined

,OrderID,CustomerID,CustomerName,Segment,RegionName,OrderAmount
0,O001,C001,Amina,Retail,Nairobi,4500
1,O002,C002,Brian,Corporate,Mombasa,12500
2,O003,C001,Amina,Retail,Nairobi,3200
3,O004,C003,Carol,Retail,Nairobi,6800
4,O005,C004,David,SME,Kisumu,9100
5,O006,C005,Esther,Corporate,Nakuru,15000
6,O007,C006,Faith,SME,Nairobi,7600
7,O008,C002,Brian,Corporate,Mombasa,8300
8,O009,C007,George,Retail,Mombasa,2500
9,O010,C008,Halima,Corporate,Kisumu,11000


In [43]:
# add final joined to the database
final_joined.to_sql(
    "final_joined",
    conn,
    if_exists="replace",
    index=False
)

15

In [44]:
# regional sales analysis
query = """ 
SELECT RegionName as Region, COUNT(OrderID) AS NumberOfOrders, SUM(OrderAmount) AS TotalSales
FROM final_joined
GROUP BY RegionName;
"""

pd.read_sql_query(query, conn)  

,Region,NumberOfOrders,TotalSales
0,None,1,9900
1,Kisumu,2,20100
2,Mombasa,3,23300
3,Nairobi,7,39800
4,Nakuru,2,20400


##### Q17 - Nairobi generated the highest sales at 39,800

##### Q18 - No management shouldnt conclude that Nairobi is the best perfoming region. It has the highest total 39,800 from 7 orders, but total sales alone doesn't show that it performs best because of the limitations of the data;
##### - The sample is to small so one or two orders can reorder the ranking. Nakuru and Kisumu have only 2 orders each, which is too few to say their averages are typical.
##### - The time period isnt shown hence If the orders come from different or uneven periods, the totals aren't comparable, and there is no trend to show whether a region is growing or declining.


### Conclusion
#### The biggest data-quality problem was the duplicate CustomerID C004 in the customers table, along with at least one order whose customer had no matching record. The duplicate caused a repeating that customer's order and inflating counts and totals, while the unmatched order came through with no Segment or Region and could not be attributed. Totals should always be checked after joining tables because joins fail silently, and comparing the sum of OrderAmount before and after confirms that no rows were duplicated or lost. The Corporate segment generated the most sales, with 46,800 from just 4 orders. One limitation is that the dataset is very small so one or two orders can reorder the ranking. Nakuru and Kisumu have only 2 orders each, which is too few to say their averages are typical.